# Baby Step 8 — Simulate Controlled Instruction and Outreach Without Sending Anything

**Author:** Alejandro Reynoso  
**Persistent vault:** `/content/drive/MyDrive/Alejandro-Reynoso-Corporate-Civil-Litigation-ExoBrain`

Baby Step 8 designs and tests the controls that must exist between an internal provider shortlist and any real instruction or communication.

The notebook:

- verifies synthetic recipient identity;
- refreshes conflict and independence status;
- assigns disclosure tiers;
- creates approved message templates;
- generates immutable contact identifiers;
- simulates responses;
- updates internal process state;
- proves that no send path exists;
- records DEC-008;
- preserves Recommendation V1;
- prohibits Recommendation V2 and all real external action.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, csv, datetime, hashlib, re
from collections import Counter, defaultdict

VAULT = Path(r"/content/drive/MyDrive/Alejandro-Reynoso-Corporate-Civil-Litigation-ExoBrain")
if not VAULT.exists():
    raise FileNotFoundError("Run Baby Step 0 first.")

state_path = VAULT/"00_System"/"Workflow_State.json"
state = json.loads(state_path.read_text(encoding="utf-8"))

if 7 not in state.get("completed_steps", []):
    raise RuntimeError("Baby Step 7 is not complete.")

shortlists = json.loads((VAULT/"data"/"baby_step_7_shortlists.json").read_text(encoding="utf-8"))
providers = json.loads((VAULT/"data"/"baby_step_7_provider_universe.json").read_text(encoding="utf-8"))
working_cases = json.loads((VAULT/"data"/"baby_step_6_working_cases.json").read_text(encoding="utf-8"))

print("Shortlist records:",len(shortlists))
print("Providers:",len(providers))


## Absolute boundary

The notebook contains no email API, no messaging API, no HTTP request, no network client, and no external connector.

All recipients use reserved `.invalid` domains.

All actions remain `DRY_RUN`.


In [ ]:
DRY_RUN = True
NETWORK_DISABLED = True
SEND_FUNCTION_PRESENT = False

if not DRY_RUN:
    raise RuntimeError("Baby Step 8 must remain in dry-run mode.")

if not NETWORK_DISABLED:
    raise RuntimeError("Network must remain disabled.")

if SEND_FUNCTION_PRESENT:
    raise RuntimeError("A send path is not permitted.")


## Disclosure tiers

| Tier | Content | Maximum state in Baby Step 8 |
|---|---|---|
| 0 | Internal-only analysis | Allowed |
| 1 | Anonymized synthetic instruction summary | Simulated only |
| 2 | Matter-identified limited package | Blocked |
| 3 | Full confidential instruction pack | Blocked |
| 4 | Live engagement and data-room access | Blocked |

The prototype simulates Tier 1 only.


In [ ]:
DISCLOSURE_TIERS = {
    0:{
        "name":"Internal Only",
        "allowed_content":["internal analysis","conflicts","working case"],
        "real_release_allowed":False
    },
    1:{
        "name":"Anonymized Synthetic Instruction Summary",
        "allowed_content":["synthetic matter type","scope request","timing request","conflict-check request"],
        "real_release_allowed":False
    },
    2:{
        "name":"Matter-Identified Limited Package",
        "allowed_content":["matter identity","selected facts","scope"],
        "real_release_allowed":False
    },
    3:{
        "name":"Confidential Instruction Pack",
        "allowed_content":["full synthetic record"],
        "real_release_allowed":False
    },
    4:{
        "name":"Live Engagement and Data Room",
        "allowed_content":["engagement","data-room access"],
        "real_release_allowed":False
    }
}

(VAULT/"00_System"/"Baby_Step_8_Disclosure_Tiers.json").write_text(
    json.dumps(DISCLOSURE_TIERS,indent=2),encoding="utf-8"
)


## Synthetic recipient registry

Only Wave 1 and Conditional Wave 2 candidates receive simulated recipient records.

Each recipient has:

- unique identity;
- fictional role;
- reserved `.invalid` domain;
- authority field;
- conflict state;
- approved wave;
- maximum disclosure tier.


In [ ]:
provider_lookup = {p["provider_id"]:p for p in providers}

selected = [
    x for x in shortlists
    if x["wave"] in ["WAVE 1","CONDITIONAL WAVE 2"]
]

recipients = []
for idx,item in enumerate(selected,1):
    p = provider_lookup[item["provider_id"]]
    role = {
        "Outside Counsel":"Synthetic Relationship Partner",
        "Expert":"Synthetic Expert Principal",
        "Vendor":"Synthetic Engagement Manager"
    }[p["provider_type"]]

    max_tier = 1 if item["wave"]=="WAVE 1" and item["eligible"] else 0

    recipients.append({
        "recipient_id":f"RCP-{idx:03d}",
        "provider_id":p["provider_id"],
        "matter_id":item["matter_id"],
        "provider_type":p["provider_type"],
        "display_name":role + " " + p["provider_id"],
        "email":f"{p['provider_id'].lower()}-{item['matter_id'].lower()}@example.invalid",
        "authority_to_receive":True if item["wave"]=="WAVE 1" else False,
        "conflict_state":item["conflict_state"],
        "independence_state":item["independence_state"],
        "approved_wave":item["wave"],
        "maximum_disclosure_tier":max_tier,
        "synthetic":True
    })

assert len({r["recipient_id"] for r in recipients})==len(recipients)
assert len({r["email"] for r in recipients})==len(recipients)
assert all(r["email"].endswith(".invalid") for r in recipients)

(VAULT/"data"/"baby_step_8_recipient_registry.json").write_text(
    json.dumps(recipients,indent=2),encoding="utf-8"
)

print("Synthetic recipients:",len(recipients))


## Contact-control protocol

A simulated instruction may proceed only if all controls pass:

1. recipient identity verified;
2. recipient authority verified;
3. conflict state refreshed;
4. approved wave confirmed;
5. disclosure tier assigned;
6. exact content approved;
7. immutable contact ID generated;
8. second human gate recorded;
9. response classified;
10. no network activity possible.


In [ ]:
CONTROL_REQUIREMENTS = [
    "identity_verified",
    "authority_verified",
    "conflict_refreshed",
    "wave_confirmed",
    "tier_valid",
    "content_approved",
    "contact_id_created",
    "second_human_gate",
    "response_classified",
    "network_disabled"
]

message_templates = {
    "Outside Counsel":(
        "Synthetic conflict and availability request concerning a fictional "
        "corporate civil matter. No instruction, engagement, or confidential "
        "information is transmitted."
    ),
    "Expert":(
        "Synthetic independence, availability, and scope request concerning a "
        "fictional dispute. No instruction or confidential information is transmitted."
    ),
    "Vendor":(
        "Synthetic capability, timing, and capacity request concerning a fictional "
        "litigation-support workstream. No engagement or data is transmitted."
    )
}


## Dry-run instruction records

The notebook simulates Tier 1 instruction records for eligible Wave 1 recipients.

Conditional Wave 2 recipients remain blocked.


In [ ]:
instruction_records = []

for recipient in recipients:
    eligible = (
        recipient["approved_wave"]=="WAVE 1"
        and recipient["authority_to_receive"]
        and recipient["maximum_disclosure_tier"]>=1
        and recipient["conflict_state"]=="CLEAR"
    )

    timestamp = datetime.datetime.now().isoformat()
    raw_id = (
        recipient["recipient_id"] + "|" +
        recipient["matter_id"] + "|" +
        timestamp
    )
    contact_id = "CNT-" + hashlib.sha256(raw_id.encode()).hexdigest()[:16]

    controls = {
        "identity_verified":True,
        "authority_verified":recipient["authority_to_receive"],
        "conflict_refreshed":recipient["conflict_state"]=="CLEAR",
        "wave_confirmed":recipient["approved_wave"]=="WAVE 1",
        "tier_valid":recipient["maximum_disclosure_tier"]>=1,
        "content_approved":eligible,
        "contact_id_created":True,
        "second_human_gate":True if eligible else False,
        "response_classified":False,
        "network_disabled":NETWORK_DISABLED
    }

    instruction_records.append({
        "contact_id":contact_id,
        "recipient_id":recipient["recipient_id"],
        "provider_id":recipient["provider_id"],
        "matter_id":recipient["matter_id"],
        "provider_type":recipient["provider_type"],
        "approved_wave":recipient["approved_wave"],
        "disclosure_tier":1 if eligible else 0,
        "message_template":message_templates[recipient["provider_type"]] if eligible else None,
        "controls":controls,
        "eligible_for_simulation":eligible,
        "simulation_status":"READY_FOR_DRY_RUN" if eligible else "BLOCKED",
        "actual_send_attempted":False,
        "actual_send_completed":False,
        "synthetic":True
    })

(VAULT/"data"/"baby_step_8_instruction_records.json").write_text(
    json.dumps(instruction_records,indent=2),encoding="utf-8"
)

print("Dry-run ready:",sum(1 for x in instruction_records if x["eligible_for_simulation"]))
print("Blocked:",sum(1 for x in instruction_records if not x["eligible_for_simulation"]))


## Simulated responses

Eligible Wave 1 recipients receive fictional response outcomes.

The responses are not evidence of real availability, conflicts, expertise, or market behavior.


In [ ]:
RESPONSE_SEQUENCE = [
    "AVAILABLE",
    "AVAILABLE_WITH_SCOPE_LIMIT",
    "CONFLICT_REVIEW_REQUIRED",
    "TIMING_HOLD",
    "AVAILABLE",
    "ADDITIONAL_INFORMATION_REQUESTED"
]

responses = []
eligible_records = [x for x in instruction_records if x["eligible_for_simulation"]]

for idx,record in enumerate(eligible_records):
    status = RESPONSE_SEQUENCE[idx % len(RESPONSE_SEQUENCE)]
    response_id = f"RSP-{idx+1:03d}"

    implications = {
        "AVAILABLE":"Candidate remains eligible for internal process design.",
        "AVAILABLE_WITH_SCOPE_LIMIT":"Scope must be narrowed before any future engagement.",
        "CONFLICT_REVIEW_REQUIRED":"Candidate moves to conditional status.",
        "TIMING_HOLD":"Candidate remains reserve pending timing refresh.",
        "ADDITIONAL_INFORMATION_REQUESTED":"No additional disclosure permitted in Baby Step 8."
    }[status]

    responses.append({
        "response_id":response_id,
        "contact_id":record["contact_id"],
        "recipient_id":record["recipient_id"],
        "provider_id":record["provider_id"],
        "matter_id":record["matter_id"],
        "response_status":status,
        "implication":implications,
        "real_response":False,
        "synthetic":True
    })

response_by_contact = {r["contact_id"]:r for r in responses}
for record in instruction_records:
    if record["contact_id"] in response_by_contact:
        record["controls"]["response_classified"]=True
        record["simulation_status"]="DRY_RUN_COMPLETE"
        record["synthetic_response_id"]=response_by_contact[record["contact_id"]]["response_id"]

(VAULT/"data"/"baby_step_8_instruction_records.json").write_text(
    json.dumps(instruction_records,indent=2),encoding="utf-8"
)
(VAULT/"data"/"baby_step_8_simulated_responses.json").write_text(
    json.dumps(responses,indent=2),encoding="utf-8"
)

print("Simulated responses:",len(responses))


## Executable control tests

Every simulated instruction must pass the complete control suite.

Any failed control blocks progression.


In [ ]:
control_results = []

for record in instruction_records:
    passed_controls = {
        key:value
        for key,value in record["controls"].items()
    }

    if record["eligible_for_simulation"]:
        all_passed = all(passed_controls.values())
    else:
        all_passed = (
            record["simulation_status"]=="BLOCKED"
            and not record["actual_send_attempted"]
            and not record["actual_send_completed"]
        )

    control_results.append({
        "contact_id":record["contact_id"],
        "eligible_for_simulation":record["eligible_for_simulation"],
        "controls":passed_controls,
        "all_required_controls_passed":all_passed,
        "actual_send_attempted":record["actual_send_attempted"],
        "actual_send_completed":record["actual_send_completed"]
    })

assert all(x["all_required_controls_passed"] for x in control_results)
assert all(not x["actual_send_attempted"] for x in control_results)
assert all(not x["actual_send_completed"] for x in control_results)

(VAULT/"data"/"baby_step_8_control_results.json").write_text(
    json.dumps(control_results,indent=2),encoding="utf-8"
)


## Process-state refresh

Synthetic responses may change only the internal planning status.

They cannot create a real engagement, instruction, or external obligation.


In [ ]:
process_updates = []

for response in responses:
    if response["response_status"]=="AVAILABLE":
        new_state="SIMULATED_WAVE_1_CONFIRMED"
    elif response["response_status"] in ["AVAILABLE_WITH_SCOPE_LIMIT","CONFLICT_REVIEW_REQUIRED"]:
        new_state="SIMULATED_CONDITIONAL"
    elif response["response_status"]=="TIMING_HOLD":
        new_state="SIMULATED_RESERVE"
    else:
        new_state="NO_FURTHER_DISCLOSURE"

    process_updates.append({
        "matter_id":response["matter_id"],
        "provider_id":response["provider_id"],
        "response_id":response["response_id"],
        "new_internal_state":new_state,
        "engagement_created":False,
        "instruction_created":False,
        "external_obligation_created":False
    })

(VAULT/"data"/"baby_step_8_process_updates.json").write_text(
    json.dumps(process_updates,indent=2),encoding="utf-8"
)


In [ ]:
def write_note(path,lines):
    path.write_text("\n".join(lines).strip()+"\n",encoding="utf-8")

outreach_dir = VAULT/"21_Simulated_Outreach"
outreach_dir.mkdir(parents=True,exist_ok=True)

for record in instruction_records:
    response = response_by_contact.get(record["contact_id"])
    lines = [
        "---",
        f"contact_id: {record['contact_id']}",
        f"matter_id: {record['matter_id']}",
        f"provider_id: {record['provider_id']}",
        "baby_step: 8",
        "dry_run: true",
        "actual_send_attempted: false",
        "actual_send_completed: false",
        "synthetic: true",
        "---","",
        f"# {record['contact_id']} — Simulated Instruction","",
        f"- Matter: [[../02_Active_Matters/{record['matter_id']}]]",
        f"- Provider: [[../19_Providers/{record['provider_id']}]]",
        f"- Wave: {record['approved_wave']}",
        f"- Disclosure tier: {record['disclosure_tier']}",
        f"- Status: **{record['simulation_status']}**","",
        "## Controls",""
    ]
    lines += [f"- {k}: {v}" for k,v in record["controls"].items()]
    lines += ["","## Message template","",
              record["message_template"] or "Blocked — no template released.",""]
    if response:
        lines += [
            "## Simulated response","",
            f"- Status: {response['response_status']}",
            f"- Implication: {response['implication']}",""
        ]
    lines += [
        "## Absolute boundary","",
        "No message was sent. No provider was contacted. No instruction or engagement was created."
    ]
    write_note(outreach_dir/f"{record['contact_id']}.md",lines)

print("Simulated outreach notes:",len(list(outreach_dir.glob("*.md"))))


## Controlled-outreach report


In [ ]:
response_counts = Counter(r["response_status"] for r in responses)

report = [
    "# Baby Step 8 — Controlled Instruction and Outreach Report","",
    "## Executive conclusion","",
    "The exo-brain successfully simulated a controlled provider-instruction workflow without sending, contacting, engaging, or transmitting anything.","",
    "## Control boundary","",
    "- DRY_RUN: true",
    "- Network disabled: true",
    "- Send function present: false",
    "- Actual sends attempted: 0",
    "- Actual sends completed: 0","",
    "## Process counts","",
    f"- Synthetic recipients: {len(recipients)}",
    f"- Dry-run eligible records: {len(eligible_records)}",
    f"- Blocked records: {len(instruction_records)-len(eligible_records)}",
    f"- Simulated responses: {len(responses)}","",
    "## Simulated response profile",""
]
report += [f"- {status}: {count}" for status,count in response_counts.items()]
report += [
    "","## Governance conclusion","",
    "The simulated response log may update internal process planning only.",
    "It does not create real availability, conflict clearance, engagement, instruction, or authority."
]
write_note(VAULT/"10_Reports"/"Baby_Step_8_Controlled_Outreach_Report.md",report)


## Human decision — DEC-008

DEC-008 accepts the dry-run protocol and synthetic response baseline.

It authorizes only the design of continuous monitoring and weekly precedent refresh.

It does not authorize real contact or Recommendation V2.


In [ ]:
DECISION = {
    "decision_id":"DEC-008",
    "date":datetime.date.today().isoformat(),
    "title":"Accept Controlled Dry-Run Instruction Protocol",
    "decision":"Accept the Baby Step 8 dry-run instruction protocol, control results, and synthetic response baseline for architecture testing only.",
    "permitted_next_actions":[
        "design continuous precedent monitoring",
        "design weekly 200-case intake",
        "design affected-matter recalculation",
        "preserve Recommendation V1"
    ],
    "not_authorized":[
        "real provider contact",
        "real provider instruction",
        "engagement letter",
        "procurement commitment",
        "network transmission",
        "Recommendation V2",
        "filing","service","party contact","court contact",
        "settlement offer","external legal advice","external distribution"
    ],
    "synthetic":True
}
(VAULT/"09_Decisions"/"DEC-008.json").write_text(json.dumps(DECISION,indent=2),encoding="utf-8")
lines = [
    "# DEC-008 — Accept Controlled Dry-Run Instruction Protocol","",
    f"**Date:** {DECISION['date']}","","## Decision","",DECISION["decision"],"",
    "## Permitted next actions",""
]
lines += [f"- {x}" for x in DECISION["permitted_next_actions"]]
lines += ["","## Not authorized",""]
lines += [f"- {x}" for x in DECISION["not_authorized"]]
write_note(VAULT/"09_Decisions"/"DEC-008.md",lines)


In [ ]:
hot = [
    "# Current State — Hot Cache","",
    "## Recommendation state","",
    "- Recommendation V1 remains preserved.",
    "- Recommendation V2 does not exist.","",
    "## Dry-run outreach state","",
    f"- Synthetic recipients: {len(recipients)}",
    f"- Dry-run completed: {len(responses)}",
    f"- Blocked: {len(instruction_records)-len(eligible_records)}",
    "- Actual sends attempted: 0",
    "- Actual sends completed: 0","",
    "## Current decision","","- [[../09_Decisions/DEC-008]]","",
    "## Permitted","",
    "- Continuous precedent-monitoring design",
    "- Weekly 200-case intake design",
    "- Affected-matter recalculation design","",
    "## Prohibited","",
    "- Real contact or instruction",
    "- Engagement letters",
    "- Network transmission",
    "- Recommendation V2","",
    "## Next permitted experiment","",
    "Ingest 200 new synthetic precedents and perform targeted recommendation-impact analysis."
]
write_note(VAULT/"12_Hot_Cache"/"Current_State.md",hot)


In [ ]:
errors = []

outreach_notes = list((VAULT/"21_Simulated_Outreach").glob("*.md"))
v1 = list((VAULT/"08_Recommendations").glob("REC-*-V001.md"))
v2 = list((VAULT/"08_Recommendations").glob("REC-*-V002.md"))

if len(outreach_notes)!=len(instruction_records):
    errors.append(f"Expected {len(instruction_records)} outreach notes, found {len(outreach_notes)}")
if len(v1)!=5:
    errors.append(f"Expected 5 Recommendation V1 notes, found {len(v1)}")
if v2:
    errors.append("Recommendation V2 exists prematurely")
if any(x["actual_send_attempted"] for x in instruction_records):
    errors.append("A send attempt was recorded")
if any(x["actual_send_completed"] for x in instruction_records):
    errors.append("A send completion was recorded")
if not NETWORK_DISABLED:
    errors.append("Network-disabled assertion failed")
if SEND_FUNCTION_PRESENT:
    errors.append("Send function must not exist")

required = [
    VAULT/"data"/"baby_step_8_recipient_registry.json",
    VAULT/"data"/"baby_step_8_instruction_records.json",
    VAULT/"data"/"baby_step_8_simulated_responses.json",
    VAULT/"data"/"baby_step_8_control_results.json",
    VAULT/"data"/"baby_step_8_process_updates.json",
    VAULT/"10_Reports"/"Baby_Step_8_Controlled_Outreach_Report.md",
    VAULT/"09_Decisions"/"DEC-008.md",
    VAULT/"09_Decisions"/"DEC-008.json"
]
for p in required:
    if not p.exists():
        errors.append(f"Missing: {p}")

validation = {
    "validated_at":datetime.datetime.now().isoformat(),
    "recipient_count":len(recipients),
    "instruction_record_count":len(instruction_records),
    "simulated_response_count":len(responses),
    "actual_send_attempted_count":sum(1 for x in instruction_records if x["actual_send_attempted"]),
    "actual_send_completed_count":sum(1 for x in instruction_records if x["actual_send_completed"]),
    "recommendation_v1_count":len(v1),
    "recommendation_v2_count":len(v2),
    "decision":"DEC-008",
    "errors":errors,
    "passed":len(errors)==0
}
(VAULT/"11_Audit"/"Baby_Step_8_Validation.json").write_text(
    json.dumps(validation,indent=2),encoding="utf-8"
)
assert validation["passed"],errors
print(json.dumps(validation,indent=2))
print("BABY STEP 8 PASSED")


In [ ]:
state.update({
    "completed_steps":sorted(set(state.get("completed_steps",[])+[8])),
    "current_step":8,
    "next_step":9,
    "decision":"DEC-008",
    "dry_run_instruction_records":len(instruction_records),
    "simulated_responses":len(responses),
    "actual_sends":0,
    "current_recommendation_version":1,
    "next_problem":"Ingest 200 new synthetic precedents and perform targeted recommendation-impact analysis.",
    "permission_state":{
        "observe":True,
        "organize":True,
        "browse":True,
        "internal_strategy_analysis":True,
        "evidence_governance":True,
        "committee_product":True,
        "controlled_internal_diligence":True,
        "remedies_and_damages_analysis":True,
        "counterparty_and_expert_selection":True,
        "simulated_instruction_design":True,
        "continuous_precedent_monitoring":True,
        "recommendation_v2":False,
        "external_action":False
    }
})
state_path.write_text(json.dumps(state,indent=2),encoding="utf-8")
audit = {
    "timestamp":datetime.datetime.now().isoformat(),
    "step":8,
    "action":"Simulated controlled provider instruction and outreach with an absolute no-send boundary.",
    "outputs":{
        "recipients":len(recipients),
        "instruction_records":len(instruction_records),
        "simulated_responses":len(responses),
        "actual_sends":0,
        "decision":"DEC-008"
    },
    "validation_passed":True
}
with (VAULT/"11_Audit"/"workflow_audit.jsonl").open("a",encoding="utf-8") as f:
    f.write(json.dumps(audit)+"\n")
